# Multi-Class Microscopic Tissue Pathology Classification
**Author:** Mehedi Hasan Rafid | **ID:** 22-48453-3
**Dataset:** PathMNIST (9-Class Colon Pathology)

### Abstract
This notebook documents the end-to-end development of a Convolutional Neural Network (CNN) pipeline to classify microscopic tissue structures. To fulfill the assignment rubric, the project progresses through three rigorous phases to maximize diagnostic accuracy:
1. **Ablation Study:** Building and evaluating a custom CNN from scratch (comparing an unregularized baseline vs. a SOTA model with Batch Normalization and 50% Dropout).
2. **Test-Time Augmentation (TTA):** Applying mathematical rotations to test data to smooth out edge-case predictions.
3. **Transfer Learning Pivot:** Deploying a pre-trained ResNet-50 architecture to overcome the dataset's intrinsic generalization barriers.

### Phase 1: Custom CNN Architecture & Hyperparameter Rationale
* **Data Handling:** 28x28 images are normalized (mean=0.5, std=0.5). MixUp augmentation is used dynamically during training to synthesize tissue boundaries.
* **Architecture:** The custom SOTA model utilizes a 3-block CNN augmented with a Convolutional Block Attention Module (CBAM) to mathematically prioritize spatial cellular structures over background noise.
* **Hyperparameters:** Adam Optimizer (lr=0.001) for adaptive momentum, paired with a CosineAnnealingLR scheduler. We utilize **Focal Loss** rather than standard Cross-Entropy to heavily penalize the model when it misclassifies hard, amorphous structures like Debris or Stroma.

In [ ]:
!pip install medmnist -q

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data
import torchvision.transforms as transforms
import medmnist
from medmnist import INFO

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using SOTA Training Environment on: {device}")

info = INFO['pathmnist']
DataClass = getattr(medmnist, info['python_class'])

data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

train_dataset = DataClass(split='train', transform=data_transform, download=True)
val_dataset = DataClass(split='val', transform=data_transform, download=True)
test_dataset = DataClass(split='test', transform=data_transform, download=True)

BATCH_SIZE = 128
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1   = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return out * x

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.sigmoid(self.conv1(out))
        return out * x

class SOTAPathCNN(nn.Module):
    def __init__(self, in_channels, num_classes, use_advanced_features=False):
        super(SOTAPathCNN, self).__init__()
        self.use_advanced = use_advanced_features

        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32) if use_advanced_features else nn.Identity()

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64) if use_advanced_features else nn.Identity()

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128) if use_advanced_features else nn.Identity()

        if use_advanced_features:
            self.ca = ChannelAttention(128)
            self.sa = SpatialAttention()

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.dropout = nn.Dropout(0.5) if use_advanced_features else nn.Identity()
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.relu(self.bn3(self.conv3(x)))

        if self.use_advanced:
            x = self.ca(x)
            x = self.sa(x)

        x = self.pool(x)
        x = self.flatten(x)
        x = self.relu(self.dropout(self.fc1(x)))
        x = self.fc2(x)
        return x

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss.sum()

def train_model(model, num_epochs=15, use_advanced=False):
    criterion = FocalLoss() if use_advanced else nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4 if use_advanced else 0)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device).squeeze()
            optimizer.zero_grad()

            if use_advanced:
                inputs, targets_a, targets_b, lam = mixup_data(inputs, targets, alpha=0.2)
                outputs = model(inputs)
                loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)

            if use_advanced:
                correct += (lam * predicted.eq(targets_a).sum().float() + (1 - lam) * predicted.eq(targets_b).sum().float()).item()
            else:
                correct += predicted.eq(targets).sum().item()

        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(100. * correct / total)

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device).squeeze()
                outputs = model(inputs)

                v_loss = F.cross_entropy(outputs, targets)
                val_loss += v_loss.item()

                _, predicted = outputs.max(1)
                val_total += targets.size(0)
                val_correct += predicted.eq(targets).sum().item()

        history['val_loss'].append(val_loss / len(val_loader))
        history['val_acc'].append(100. * val_correct / val_total)
        scheduler.step()

        print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {history['train_loss'][-1]:.4f} | Val Acc: {history['val_acc'][-1]:.2f}%")

    return history

print("\n--- Training BASELINE CNN (No BN, No Dropout, Standard CE Loss) ---")
model_base = SOTAPathCNN(in_channels=info['n_channels'], num_classes=len(info['label']), use_advanced_features=False).to(device)
history_base = train_model(model_base, num_epochs=12, use_advanced=False)

print("\n--- Training SOTA CNN (CBAM Attention, Focal Loss, MixUp, BN & Dropout) ---")
model_sota = SOTAPathCNN(in_channels=info['n_channels'], num_classes=len(info['label']), use_advanced_features=True).to(device)
history_sota = train_model(model_sota, num_epochs=12, use_advanced=True)


def plot_curves(history_base, history_sota):
    epochs = range(1, len(history_base['train_acc']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, history_base['val_acc'], 'r--', label='Baseline Val Acc')
    axes[0].plot(epochs, history_sota['val_acc'], 'b-', linewidth=2, label='SOTA Val Acc')
    axes[0].set_title('Validation Accuracy Comparison')
    axes[0].set_xlabel('Epochs')
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].legend()

    axes[1].plot(epochs, history_base['val_loss'], 'r--', label='Baseline Val Loss')
    axes[1].plot(epochs, history_sota['val_loss'], 'b-', linewidth=2, label='SOTA Val Loss')
    axes[1].set_title('Validation Loss Comparison')
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_curves(history_base, history_sota)

print("\nEvaluating the SOTA Model on the Test Set...")
model_sota.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device).squeeze()
        outputs = model_sota(inputs)
        _, predicted = outputs.max(1)
        y_true.extend(targets.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

labels = [str(i) for i in range(len(info['label']))]
print("\n--- SOTA Evaluation Metrics ---")
print(classification_report(y_true, y_pred, target_names=labels))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='magma', xticklabels=labels, yticklabels=labels)
plt.title('SOTA Confusion Matrix on Test Data')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

save_path = 'SOTA_model_weights.pth'
torch.save(model_sota.state_dict(), save_path)
print(f"\nSOTA Model weights successfully saved to {save_path}")

### Phase 1 Analysis: The Generalization Wall
The ablation study successfully proves the necessity of regularization. The baseline CNN plateaued at 93.98% validation accuracy, while the SOTA CNN (with CBAM, MixUp, and Focal Loss) achieved a superior **96.61%**.

However, on the withheld test set, the overall accuracy dropped to **88%**. The model is highly reliable at identifying malignant tumors (Class 8) with a 92% recall rate. However, it struggles heavily with Cancer-Associated Stroma (Class 7), achieving a recall of only 39%.

### Phase 2 Rationale: Test-Time Augmentation (TTA)
To attempt to close this generalization gap without increasing model size, Phase 2 implements Test-Time Augmentation. By applying random horizontal flips and 15-degree rotations to the test images during inference, we aim to force the CNN to evaluate ambiguous slides from multiple angles, mimicking a real pathologist.

In [ ]:
import torch.nn.functional as F

print("\n--- Running Test-Time Augmentation (TTA) ---")

tta_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

tta_dataset = DataClass(split='test', transform=tta_transforms, download=True)
tta_loader = data.DataLoader(dataset=tta_dataset, batch_size=BATCH_SIZE, shuffle=False)

model_sota.eval()
all_standard_preds = []
all_tta_preds = []
y_true = []

with torch.no_grad():
    for (inputs, targets), (tta_inputs, _) in zip(test_loader, tta_loader):
        inputs, targets = inputs.to(device), targets.to(device).squeeze()
        tta_inputs = tta_inputs.to(device)

        outputs_standard = F.softmax(model_sota(inputs), dim=1)

        outputs_tta = F.softmax(model_sota(tta_inputs), dim=1)

        ensemble_outputs = (outputs_standard + outputs_tta) / 2.0

        _, predicted = ensemble_outputs.max(1)

        y_true.extend(targets.cpu().numpy())
        all_tta_preds.extend(predicted.cpu().numpy())

print("\n--- Evaluation Metrics WITH Test-Time Augmentation ---")
print(classification_report(y_true, all_tta_preds, target_names=labels))

### Phase 2 Analysis: The TTA Tradeoff
Contrary to expectations, TTA decreased the overall test accuracy from 88% to **86%**. Analyzing the per-class metrics reveals why:
* **Amorphous Tissues Benefited:** The recall for Class 2 (Debris) remained stable, and precision jumped to 66%. Because debris lacks a geometric orientation, viewing it from multiple augmented angles improved the model's confidence.
* **Structured Tissues Degraded:** The recall for Class 7 (Stroma) collapsed to **24%**. Stroma serves as the connective framework surrounding tumors and possesses a specific directional orientation. Randomly rotating these images destroyed the spatial context the CNN relied upon.

### Phase 3 Rationale: Transfer Learning (ResNet-50)
Because the custom 28x28 CNN has hit a feature-extraction ceiling, Phase 3 implements Transfer Learning. We dynamically upscale the images to 224x224 and utilize a pre-trained **ResNet-50** (ImageNet1K_V2 weights). The massive 50-layer depth is mathematically proven to extract highly complex biological textures that a shallow network cannot.

In [ ]:
import torchvision.models as models

print("\n--- Initializing Transfer Learning Pipeline (ResNet-50) ---")

tl_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

tl_train_dataset = DataClass(split='train', transform=tl_transform, download=True)
tl_val_dataset = DataClass(split='val', transform=tl_transform, download=True)
tl_test_dataset = DataClass(split='test', transform=tl_transform, download=True)

TL_BATCH_SIZE = 64
tl_train_loader = data.DataLoader(dataset=tl_train_dataset, batch_size=TL_BATCH_SIZE, shuffle=True)
tl_val_loader = data.DataLoader(dataset=tl_val_dataset, batch_size=TL_BATCH_SIZE, shuffle=False)
tl_test_loader = data.DataLoader(dataset=tl_test_dataset, batch_size=TL_BATCH_SIZE, shuffle=False)

model_resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

num_ftrs = model_resnet.fc.in_features
model_resnet.fc = nn.Linear(num_ftrs, len(info['label']))
model_resnet = model_resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_resnet = optim.Adam(model_resnet.parameters(), lr=0.0001)
scheduler_resnet = optim.lr_scheduler.StepLR(optimizer_resnet, step_size=3, gamma=0.5)

from tqdm import tqdm

num_tl_epochs = 5
for epoch in range(num_tl_epochs):
    model_resnet.train()
    train_loss, correct, total = 0, 0, 0

    train_loop = tqdm(tl_train_loader, desc=f"Epoch [{epoch+1}/{num_tl_epochs}] Training")

    for inputs, targets in train_loop:
        inputs, targets = inputs.to(device), targets.to(device).squeeze()

        optimizer_resnet.zero_grad()
        outputs = model_resnet(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer_resnet.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        train_loop.set_postfix(loss=train_loss/total)

    val_loss, val_correct, val_total = 0, 0, 0
    model_resnet.eval()

    val_loop = tqdm(tl_val_loader, desc=f"Epoch [{epoch+1}/{num_tl_epochs}] Validation")
    with torch.no_grad():
        for inputs, targets in val_loop:
            inputs, targets = inputs.to(device), targets.to(device).squeeze()
            outputs = model_resnet(inputs)
            v_loss = criterion(outputs, targets)
            val_loss += v_loss.item()
            _, predicted = outputs.max(1)
            val_total += targets.size(0)
            val_correct += predicted.eq(targets).sum().item()

    scheduler_resnet.step()
    print(f"\nResult Epoch [{epoch+1}/{num_tl_epochs}] - Train Acc: {100.*correct/total:.2f}% | Val Acc: {100.*val_correct/val_total:.2f}%\n")

print("\nEvaluating ResNet-50 on the Test Set...")
model_resnet.eval()
y_true_tl, y_pred_tl = [], []
with torch.no_grad():
    for inputs, targets in tl_test_loader:
        inputs, targets = inputs.to(device), targets.to(device).squeeze()
        outputs = model_resnet(inputs)
        _, predicted = outputs.max(1)
        y_true_tl.extend(targets.cpu().numpy())
        y_pred_tl.extend(predicted.cpu().numpy())

print("\n--- ResNet-50 Transfer Learning Metrics ---")
print(classification_report(y_true_tl, y_pred_tl, target_names=labels))

torch.save(model_resnet.state_dict(), 'ResNet50_PathMNIST_weights.pth')
print("\nResNet-50 Model weights successfully saved.")

In [ ]:
!pip install torchinfo -q
from torchinfo import summary

print("\n--- ResNet-50 Transfer Learning Architecture ---")
architecture_summary = summary(model_resnet, input_size=(64, 3, 224, 224),
                               col_names=["input_size", "output_size", "num_params", "trainable"],
                               col_width=20, row_settings=["var_names"])
print(architecture_summary)

In [ ]:
!pip install torchviz -q
import torch
import torch.nn as nn
from torchviz import make_dot

class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(10, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNN()

dummy_input = torch.randn(1, 10)

output = model(dummy_input)

viz = make_dot(output, params=dict(model.named_parameters()))
viz.render("model_architecture", format="png")

viz

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import torch.nn.functional as F

print("\n--- Generating Multi-Class ROC-AUC Curves ---")

model_resnet.eval()
y_true_all = []
y_probs_all = []

with torch.no_grad():
    for inputs, targets in tl_test_loader:
        inputs, targets = inputs.to(device), targets.to(device).squeeze()
        outputs = model_resnet(inputs)
        probs = F.softmax(outputs, dim=1)

        y_true_all.extend(targets.cpu().numpy())
        y_probs_all.extend(probs.cpu().numpy())

y_true_all = np.array(y_true_all)
y_probs_all = np.array(y_probs_all)

n_classes = len(info['label'])
y_true_bin = label_binarize(y_true_all, classes=range(n_classes))

fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_probs_all[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(10, 8))
colors = sns.color_palette("husl", n_classes)

for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'Class {i} (AUC = {roc_auc[i]:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Multi-Class ROC-AUC for ResNet-50 (PathMNIST)', fontsize=14)
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print("\n--- Extracting Misclassified Edge Cases ---")

misclassified_idx = np.where(np.array(y_pred_tl) != np.array(y_true_tl))[0]

def imshow_denorm(img_tensor):
    img = img_tensor.cpu().numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)
    return img

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

current_idx = 0
found = 0

for inputs, targets in tl_test_loader:
    if found >= 6:
        break
    for i in range(inputs.size(0)):
        if current_idx in misclassified_idx:
            img = imshow_denorm(inputs[i])
            true_label = targets[i].item()
            pred_label = y_pred_tl[current_idx]

            axes[found].imshow(img)
            axes[found].set_title(f"True: Class {true_label} | Pred: Class {pred_label}",
                                  color="red" if true_label != pred_label else "black", fontsize=11)
            axes[found].axis('off')
            found += 1
            if found >= 6:
                break
        current_idx += 1

plt.suptitle("Sample Misclassifications (Model Blind Spots)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Conclusions & Future Work

### Conclusion
The pivot to Transfer Learning was a resounding success, elevating the overall test accuracy to a state-of-the-art **92%**. The deep convolutional layers of ResNet-50 successfully mapped the previously problematic amorphous textures, increasing the F1-score of Debris to an exceptional **0.87**. Furthermore, the model demonstrated extreme clinical reliability in identifying malignant tissues, achieving a **97% recall rate** for Colorectal Adenocarcinoma.

### Future Work
While the current architecture is highly robust, pushing the performance closer to 99% clinical perfection requires addressing the final bottleneck: Cancer-Associated Stroma (Class 7). Future research will focus on replacing the ResNet-50 backbone with a **Swin Transformer**. By leveraging self-attention mechanisms, the model will be able to understand the global spatial relationship of Stroma relative to surrounding malignant cells, rather than relying purely on localized convolutions.